<a href="https://colab.research.google.com/github/wamo12/FinRL/blob/master/IRRBB_Repricing_Gap_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IRRBB Repricing Gap, ΔNII and ΔEVE Engine
**Production-style notebook for Google Colab** (extends the simple six-account gap exercise)

| Item | Default |
|---|---|
| Bank profile | USD 25bn retail-heavy bank, single currency (USD) |
| Buckets | 19 BCBS 368 standardised time bands |
| Measures | Repricing gap, 12M ΔNII (with deposit betas), ΔEVE, BCBS outlier test |
| NMD treatment | Core / non-core split, BCBS 368 caps (5.0y / 4.5y / 4.0y average maturity) |
| Behavioural | Mortgage and consumer prepayment (CPR), term-deposit early withdrawal |
| Inputs | Synthetic portfolio, or upload your own CSV/Excel in the same schema |

**Pipeline:** config → portfolio (synthetic or upload) → standardise → validate → repricing-flow engine → gap analytics → ΔNII / ΔEVE → limits → charts → export → tests.

> All data and market rates are **synthetic and illustrative**. This is a modelling framework, not a regulatory submission.

## 1. Setup and configuration

In [1]:
import logging, warnings
from dataclasses import dataclass, field, asdict
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore", category=FutureWarning)
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
log = logging.getLogger("irrbb")
pd.options.display.float_format = "{:,.2f}".format
pd.options.display.width = 220
pd.options.display.max_columns = 40


@dataclass
class Config:
    report_date: str = "2026-09-30"
    currency: str = "USD"
    units: str = "USD mm"
    seed: int = 42
    total_assets: float = 25_000.0          # GL total assets (used for reconciliation and % metrics)
    tier1_capital: float = 1_450.0
    use_uploaded_file: bool = False         # True -> prompts for CSV/XLSX upload in Colab
    gl_check: bool = True                   # reconcile non-derivative balances to total_assets
    output_dir: str = "irrbb_outputs"

    # Illustrative zero curve (continuously compounded, decimal)
    curve_tenors: tuple = (1/12, 0.25, 0.5, 1, 2, 3, 5, 7, 10, 20, 30)
    curve_rates: tuple = (0.0410, 0.0400, 0.0390, 0.0375, 0.0365, 0.0370, 0.0385, 0.0400, 0.0415, 0.0445, 0.0440)

    # BCBS 368 USD shock sizes (decimal)
    shock_parallel: float = 0.02
    shock_short: float = 0.03
    shock_long: float = 0.015
    apply_floor: bool = True                # BCBS floor: -150bp at t=0 rising 3bp/yr to 0 at 50y
    use_deposit_beta: bool = True

    # NMD assumptions: core share, average maturity (years), betas, deposit rate as share of short rate
    nmd: dict = field(default_factory=lambda: {
        "retail_transactional":    dict(core=0.80, avg_mat=5.0, cap_core=0.90, cap_mat=5.0, beta_up=0.10, beta_down=0.08, rate_ratio=0.05),
        "retail_nontransactional": dict(core=0.65, avg_mat=4.0, cap_core=0.70, cap_mat=4.5, beta_up=0.45, beta_down=0.35, rate_ratio=0.45),
        "wholesale":               dict(core=0.45, avg_mat=3.5, cap_core=0.50, cap_mat=4.0, beta_up=0.35, beta_down=0.30, rate_ratio=0.35),
    })

    # Limits (negative = loss)
    nii_limit_pct: float = -0.10            # dNII / baseline NII
    eve_limit_pct: float = -0.15            # dEVE / Tier 1  (BCBS supervisory outlier test)
    amber_fraction: float = 0.80            # amber when usage > 80% of limit


cfg = Config()
Path(cfg.output_dir).mkdir(exist_ok=True)
REPORT_DATE = pd.Timestamp(cfg.report_date)
MAX_MONTHS = 360   # cap for contractual runoff grid (30y)

## 2. BCBS 368 time buckets and market curve
Nineteen standardised bands with BCBS midpoints. Bucket rule: lower < t ≤ upper, with t in years.

In [2]:
BUCKETS = pd.DataFrame({
    "label": ["O/N", "O/N-1M", "1M-3M", "3M-6M", "6M-9M", "9M-1Y", "1Y-1.5Y", "1.5Y-2Y", "2Y-3Y", "3Y-4Y",
              "4Y-5Y", "5Y-6Y", "6Y-7Y", "7Y-8Y", "8Y-9Y", "9Y-10Y", "10Y-15Y", "15Y-20Y", "20Y+"],
    "upper": [1/365.25, 1/12, 0.25, 0.5, 0.75, 1, 1.5, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 20, np.inf],
    "mid":   [0.0028, 0.0417, 0.1667, 0.375, 0.625, 0.875, 1.25, 1.75, 2.5, 3.5, 4.5, 5.5, 6.5, 7.5, 8.5, 9.5, 12.5, 17.5, 25.0],
})
BL = BUCKETS["label"].tolist()
NB = len(BL)
UPPER = BUCKETS["upper"].values
MID = BUCKETS["mid"].values


def assign_bucket(t_years):
    """Map time in years to bucket index (lower < t <= upper)."""
    return np.digitize(np.asarray(t_years, dtype=float), UPPER[:-1], right=True)


def zero_rate(t, cfg=cfg):
    return np.interp(t, cfg.curve_tenors, cfg.curve_rates)

BUCKETS

,label,upper,mid
0,O/N,0.00,0.00
1,O/N-1M,0.08,0.04
2,1M-3M,0.25,0.17
3,3M-6M,0.50,0.38
4,6M-9M,0.75,0.62
5,9M-1Y,1.00,0.88
6,1Y-1.5Y,1.50,1.25
7,1.5Y-2Y,2.00,1.75
8,2Y-3Y,3.00,2.50
9,3Y-4Y,4.00,3.50


## 3. Portfolio schema and synthetic generator
Required schema (also used for uploads). `rate_type` ∈ {fixed, floating, nmd, nonsens}; `amort` ∈ {bullet, amort}; `side` ∈ {A, L}.
Balances in USD mm. Derivatives enter as two legs (`is_derivative=True`) and are excluded from balance-sheet reconciliation.

In [3]:
REQUIRED_COLS = ["instrument_id", "product", "side", "rate_type", "balance", "coupon"]
OPTIONAL_DEFAULTS = {"category": "Other", "amort": "bullet", "rem_months": np.nan, "reset_days": np.nan, "cpr": 0.0,
                     "nmd_segment": "", "is_derivative": False}

# product, side, category, total (USD mm), n accounts, rate_type, amort, term months (lo, hi), reset days (lo, hi),
# spread, reference tenor (y), annual CPR / early-withdrawal rate, NMD segment, balance dispersion
PRODUCT_SPEC = [
    ("Cash & central bank reserves", "A", "Liquidity",  1500, 5,    "floating", "bullet", (1, 1),    (1, 1),   -0.0010, 1/12, 0.00, "", 0.5),
    ("Interbank placements",         "A", "Liquidity",   500, 15,   "fixed",    "bullet", (1, 3),    (1, 1),    0.0020, 0.25, 0.00, "", 0.6),
    ("Fixed-rate mortgages",         "A", "Loans",      6500, 2500, "fixed",    "amort",  (60, 348), (1, 1),    0.0180, 7,    0.08, "", 0.7),
    ("Adjustable-rate mortgages",    "A", "Loans",      3000, 800,  "floating", "bullet", (120, 348),(1, 365),  0.0220, 1,    0.00, "", 0.7),
    ("Consumer loans",               "A", "Loans",      1800, 1200, "fixed",    "amort",  (12, 84),  (1, 1),    0.0500, 3,    0.05, "", 0.8),
    ("Corporate loans - floating",   "A", "Loans",      4500, 150,  "floating", "bullet", (12, 84),  (1, 90),   0.0200, 0.25, 0.00, "", 1.1),
    ("Corporate loans - fixed",      "A", "Loans",      2000, 80,   "fixed",    "bullet", (12, 120), (1, 1),    0.0250, 5,    0.00, "", 1.1),
    ("Securities HTM",               "A", "Securities", 2200, 40,   "fixed",    "bullet", (24, 180), (1, 1),    0.0040, 7,    0.00, "", 0.9),
    ("Securities AFS",               "A", "Securities", 1800, 40,   "fixed",    "bullet", (12, 120), (1, 1),    0.0020, 5,    0.00, "", 0.9),
    ("Other non-rate-sensitive assets", "A", "Other",   1200, 1,    "nonsens",  "bullet", (1, 1),    (1, 1),    0.0,    0.25, 0.00, "", 0.1),
    ("Retail transactional deposits",   "L", "Deposits", 3500, 600, "nmd",      "bullet", (1, 1),    (1, 1),    0.0,    1/12, 0.00, "retail_transactional", 0.9),
    ("Retail savings deposits",         "L", "Deposits", 4500, 900, "nmd",      "bullet", (1, 1),    (1, 1),    0.0,    1/12, 0.00, "retail_nontransactional", 0.9),
    ("Corporate operational deposits",  "L", "Deposits", 3500, 150, "nmd",      "bullet", (1, 1),    (1, 1),    0.0,    1/12, 0.00, "wholesale", 1.2),
    ("Retail term deposits",            "L", "Deposits", 4500, 1000,"fixed",    "bullet", (1, 60),   (1, 1),   -0.0020, 1,    0.05, "", 0.7),
    ("Corporate / brokered term deposits", "L", "Deposits", 1500, 150, "fixed", "bullet", (1, 36),   (1, 1),    0.0010, 0.5,  0.02, "", 1.0),
    ("Repo & interbank borrowing",      "L", "Wholesale", 1200, 8,  "floating", "bullet", (1, 6),    (1, 30),   0.0010, 1/12, 0.00, "", 0.6),
    ("FHLB advances",                   "L", "Wholesale", 1800, 12, "fixed",    "bullet", (12, 60),  (1, 1),    0.0030, 3,    0.00, "", 0.6),
    ("Senior & covered bonds",          "L", "Wholesale", 2000, 10, "fixed",    "bullet", (36, 120), (1, 1),    0.0080, 7,    0.00, "", 0.5),
    ("Subordinated debt",               "L", "Wholesale",  400, 3,  "fixed",    "bullet", (84, 120), (1, 1),    0.0250, 10,   0.00, "", 0.3),
    ("Other non-rate-sensitive liabilities", "L", "Other", 600, 1,  "nonsens",  "bullet", (1, 1),    (1, 1),    0.0,    0.25, 0.00, "", 0.1),
    ("Equity", "L", "Other", 1500, 1, "nonsens", "bullet", (1, 1), (1, 1), 0.0, 0.25, 0.00, "", 0.1),
]
SPEC_COLS = ["product", "side", "category", "total", "n", "rate_type", "amort", "term", "reset", "spread", "ref", "cpr", "nmd_seg", "sigma"]


def generate_portfolio(cfg: Config) -> pd.DataFrame:
    rng = np.random.default_rng(cfg.seed)
    rows = []
    k = 0
    for spec in PRODUCT_SPEC:
        s = dict(zip(SPEC_COLS, spec))
        n = s["n"]
        w = rng.lognormal(0, s["sigma"], n)
        bal = w / w.sum() * s["total"]
        rem = rng.integers(s["term"][0], s["term"][1] + 1, n)
        rst = rng.integers(s["reset"][0], s["reset"][1] + 1, n)
        noise = rng.normal(0, 0.0015, n)
        if s["rate_type"] == "nmd":
            cpn = zero_rate(1/12, cfg) * cfg.nmd[s["nmd_seg"]]["rate_ratio"] + noise * 0.2
        elif s["rate_type"] == "nonsens":
            cpn = np.zeros(n)
        else:
            cpn = zero_rate(s["ref"], cfg) + s["spread"] + noise
        for i in range(n):
            k += 1
            rows.append(dict(instrument_id=f"{s['side']}{k:05d}", product=s["product"], category=s["category"], side=s["side"],
                             rate_type=s["rate_type"], amort=s["amort"], balance=bal[i], coupon=max(cpn[i], 0.0),
                             rem_months=int(rem[i]), reset_days=int(rst[i]), cpr=s["cpr"], nmd_segment=s["nmd_seg"],
                             is_derivative=False))
    # Interest-rate swaps: 2 legs each. Pay-fixed (hedging fixed mortgages) and receive-fixed (hedging floating loans)
    swaps = [("Swap pay-fixed", 1500, 6, "L", "A"), ("Swap receive-fixed", 800, 4, "A", "L")]
    for name, notional, n, fixed_side, float_side in swaps:
        w = rng.lognormal(0, 0.4, n); nots = w / w.sum() * notional
        terms = rng.integers(12, 73, n); rsts = rng.integers(1, 91, n)
        for i in range(n):
            fx = zero_rate(terms[i] / 12, cfg)
            for leg, side, rt, cpn in (("fixed leg", fixed_side, "fixed", fx), ("float leg", float_side, "floating", zero_rate(0.25, cfg))):
                k += 1
                rows.append(dict(instrument_id=f"D{k:05d}", product=f"{name} - {leg}", category="Derivatives", side=side,
                                 rate_type=rt, amort="bullet", balance=nots[i], coupon=cpn, rem_months=int(terms[i]),
                                 reset_days=int(rsts[i]), cpr=0.0, nmd_segment="", is_derivative=True))
    df = pd.DataFrame(rows)
    return df

portfolio_raw = generate_portfolio(cfg)
log.info("Generated %d instruments", len(portfolio_raw))

## 4. Loading, standardisation and data-quality validation

In [4]:
def load_uploaded_portfolio() -> pd.DataFrame:
    from google.colab import files  # Colab only
    up = files.upload()
    name = next(iter(up))
    return pd.read_csv(name) if name.lower().endswith(".csv") else pd.read_excel(name)


def standardise(df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    """Add defaults, derive rem_months / reset_days from dates when provided, set derived fields."""
    df = df.copy()
    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    for c, v in OPTIONAL_DEFAULTS.items():
        if c not in df.columns:
            df[c] = v
    if "maturity_date" in df.columns:
        md_ = pd.to_datetime(df["maturity_date"], errors="coerce")
        df["rem_months"] = df["rem_months"].fillna(((md_ - REPORT_DATE).dt.days / 30.4375).round())
    if "next_reset_date" in df.columns:
        nr = pd.to_datetime(df["next_reset_date"], errors="coerce")
        df["reset_days"] = df["reset_days"].fillna((nr - REPORT_DATE).dt.days)
    df["side"] = df["side"].astype(str).str.upper().str.strip()
    df["rate_type"] = df["rate_type"].astype(str).str.lower().str.strip()
    df["amort"] = df["amort"].astype(str).str.lower().str.strip()
    df["nmd_segment"] = df["nmd_segment"].fillna("").astype(str)
    df["is_derivative"] = df["is_derivative"].astype(bool)
    df["rate_sensitive"] = df["rate_type"] != "nonsens"
    df["maturity_date"] = REPORT_DATE + pd.to_timedelta(df["rem_months"].fillna(0) * 30.4375, unit="D")
    df["next_reset_date"] = REPORT_DATE + pd.to_timedelta(df["reset_days"].fillna(0), unit="D")
    return df


def validate(df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    """Return exceptions table (instrument_id, severity, rule). Raises on any ERROR."""
    ex = []

    def add(mask, sev, rule):
        if mask.any():
            ex.append(pd.DataFrame({"instrument_id": df.loc[mask, "instrument_id"], "severity": sev, "rule": rule}))

    add(df["instrument_id"].duplicated(keep=False), "ERROR", "duplicate instrument_id")
    add(df[REQUIRED_COLS].isna().any(axis=1), "ERROR", "null in required field")
    add(~(df["balance"] > 0), "ERROR", "non-positive balance")
    add(~df["side"].isin(["A", "L"]), "ERROR", "side must be A or L")
    add(~df["rate_type"].isin(["fixed", "floating", "nmd", "nonsens"]), "ERROR", "invalid rate_type")
    add(~df["amort"].isin(["bullet", "amort"]), "ERROR", "invalid amort")
    add((df["rate_type"] == "fixed") & ~(df["rem_months"] >= 1), "ERROR", "fixed instrument needs rem_months >= 1")
    add((df["rate_type"] == "floating") & ~(df["reset_days"] >= 0), "ERROR", "floating instrument needs reset_days >= 0")
    add((df["rate_type"] == "nmd") & ~df["nmd_segment"].isin(cfg.nmd.keys()), "ERROR", "unknown nmd_segment")
    add((df["rate_type"] == "nmd") & (df["side"] != "L"), "ERROR", "NMD must be a liability")
    add(~df["cpr"].between(0, 1), "ERROR", "cpr outside [0,1]")
    add(~df["coupon"].between(-0.02, 0.25), "WARN", "coupon outside [-2%, 25%]")
    add((df["rate_type"] == "fixed") & (df["rem_months"] > MAX_MONTHS), "WARN", f"rem_months capped at {MAX_MONTHS}")
    out = pd.concat(ex, ignore_index=True) if ex else pd.DataFrame(columns=["instrument_id", "severity", "rule"])

    for seg, p in cfg.nmd.items():   # regulatory caps on NMD assumptions
        if p["core"] > p["cap_core"] or p["avg_mat"] > p["cap_mat"]:
            out.loc[len(out)] = [seg, "ERROR", "NMD assumption breaches BCBS cap"]

    if cfg.gl_check:                 # reconciliation to general ledger
        nd = df[~df["is_derivative"]]
        a, l = nd.loc[nd.side == "A", "balance"].sum(), nd.loc[nd.side == "L", "balance"].sum()
        if abs(a - cfg.total_assets) > 0.01:
            out.loc[len(out)] = ["GL", "ERROR", f"assets {a:,.2f} != GL {cfg.total_assets:,.2f}"]
        if abs(a - l) > 0.01:
            out.loc[len(out)] = ["GL", "ERROR", f"assets {a:,.2f} != liabilities+equity {l:,.2f}"]
    if (out["severity"] == "ERROR").any():
        raise ValueError("Validation errors:\n" + out[out.severity == "ERROR"].head(20).to_string())
    log.info("Validation passed: %d warnings", (out["severity"] == "WARN").sum())
    return out


raw = load_uploaded_portfolio() if cfg.use_uploaded_file else portfolio_raw
portfolio = standardise(raw, cfg)
exceptions = validate(portfolio, cfg)
portfolio.head()

,instrument_id,product,category,side,rate_type,amort,balance,coupon,rem_months,reset_days,cpr,nmd_segment,is_derivative,rate_sensitive,maturity_date,next_reset_date
0,A00001,Cash & central bank reserves,Liquidity,A,floating,bullet,336.46,0.04,1,1,0.00,,False,True,2026-10-30 10:30:00,2026-10-01
1,A00002,Cash & central bank reserves,Liquidity,A,floating,bullet,171.77,0.04,1,1,0.00,,False,True,2026-10-30 10:30:00,2026-10-01
2,A00003,Cash & central bank reserves,Liquidity,A,floating,bullet,420.46,0.04,1,1,0.00,,False,True,2026-10-30 10:30:00,2026-10-01
3,A00004,Cash & central bank reserves,Liquidity,A,floating,bullet,462.39,0.04,1,1,0.00,,False,True,2026-10-30 10:30:00,2026-10-01
4,A00005,Cash & central bank reserves,Liquidity,A,floating,bullet,108.92,0.04,1,1,0.00,,False,True,2026-10-30 10:30:00,2026-10-01


In [5]:
nd = portfolio[~portfolio.is_derivative]
bs = (nd.groupby(["side", "category", "product"], sort=False)
        .agg(accounts=("balance", "size"), balance=("balance", "sum"),
             wtd_coupon=("coupon", lambda s: np.average(s, weights=nd.loc[s.index, "balance"])))
        .reset_index())
bs["wtd_coupon"] *= 100
bs.rename(columns={"wtd_coupon": "wtd_coupon_%"})

,side,category,product,accounts,balance,wtd_coupon_%
0,A,Liquidity,Cash & central bank reserves,5,"1,500.00",3.94
1,A,Liquidity,Interbank placements,15,500.00,4.22
2,A,Loans,Fixed-rate mortgages,2500,"6,500.00",5.80
3,A,Loans,Adjustable-rate mortgages,800,"3,000.00",5.95
4,A,Loans,Consumer loans,1200,"1,800.00",8.70
5,A,Loans,Corporate loans - floating,150,"4,500.00",6.00
6,A,Loans,Corporate loans - fixed,80,"2,000.00",6.32
7,A,Securities,Securities HTM,40,"2,200.00",4.37
8,A,Securities,Securities AFS,40,"1,800.00",4.01
9,A,Other,Other non-rate-sensitive assets,1,"1,200.00",0.00


## 5. Repricing-flow engine
Each instrument is converted to an `N × 19` matrix of principal amounts that **reprice** in each bucket:

- **Floating**: whole balance reprices at the next reset date.
- **Fixed bullet**: balance reprices at maturity; early-withdrawal (term deposits) accelerates a share through a monthly survival curve.
- **Fixed amortising**: level-pay schedule combined with SMM prepayment, i.e. balance \(S_m = \text{sched}_m (1-\text{SMM})^m\), where \(\text{SMM}=1-(1-\text{CPR})^{1/12}\).
- **NMD**: non-core share in O/N; core share spread evenly over \(2\times\) average maturity, so the mean repricing is the assumed average maturity.

In [6]:
def _survival_matrix(rem, coupon, amort, cpr, M=MAX_MONTHS):
    """Fraction of original balance outstanding after month m (m=0..M); zero from maturity."""
    n = np.minimum(rem.astype(int), M)[:, None]
    m = np.arange(0, M + 1)[None, :]
    r = (coupon / 12.0)[:, None]
    am = (amort == "amort")[:, None]
    with np.errstate(divide="ignore", invalid="ignore"):
        level = (1 - (1 + r) ** (-np.clip(n - m, 0, None))) / (1 - (1 + r) ** (-n))
    linear = 1 - m / n
    sched = np.where(am, np.where(r > 1e-9, level, linear), 1.0)
    smm = (1 - (1 - cpr) ** (1 / 12))[:, None]
    S = sched * (1 - smm) ** m
    S = np.where(m >= n, 0.0, S)
    S[:, 0] = 1.0
    return S


def build_flows(df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    """Return one row per rate-sensitive instrument with repricing amounts by BCBS bucket."""
    s = df[df["rate_sensitive"]].reset_index(drop=True)
    N = len(s)
    F = np.zeros((N, NB))
    bal = s["balance"].values

    fl = np.where(s["rate_type"] == "floating")[0]
    if len(fl):
        F[fl, assign_bucket(s.loc[fl, "reset_days"].values / 365.25)] = bal[fl]

    fx = np.where(s["rate_type"] == "fixed")[0]
    if len(fx):
        sub = s.iloc[fx]
        S = _survival_matrix(sub["rem_months"].values, sub["coupon"].values, sub["amort"].values, sub["cpr"].values)
        flow = (S[:, :-1] - S[:, 1:]) * sub["balance"].values[:, None]
        idx = assign_bucket(np.arange(1, MAX_MONTHS + 1) / 12)
        Fs = np.zeros((len(fx), NB))
        for b in np.unique(idx):
            Fs[:, b] = flow[:, idx == b].sum(axis=1)
        F[fx] = Fs

    for seg, p in cfg.nmd.items():
        rows = np.where((s["rate_type"] == "nmd") & (s["nmd_segment"] == seg))[0]
        if len(rows) == 0:
            continue
        months = np.arange(1, int(round(2 * p["avg_mat"] * 12)) + 1)
        prof = np.zeros(NB)
        np.add.at(prof, assign_bucket(months / 12), 1 / len(months))
        F[rows] = np.outer(bal[rows], p["core"] * prof)
        F[rows, 0] += bal[rows] * (1 - p["core"])

    meta = s[["instrument_id", "product", "category", "side", "rate_type", "nmd_segment", "is_derivative", "balance", "coupon"]]
    return pd.concat([meta, pd.DataFrame(F, columns=BL)], axis=1)


flows = build_flows(portfolio, cfg)
flows.shape

(7683, 28)

## 6. Repricing gap analytics

In [7]:
def gap_table(flows: pd.DataFrame, cfg: Config, mask=None) -> pd.DataFrame:
    f = flows if mask is None else flows[mask]
    rsa = f.loc[f.side == "A", BL].sum()
    rsl = f.loc[f.side == "L", BL].sum()
    out = pd.DataFrame({"RSA": rsa, "RSL": rsl})
    out["Gap"] = out["RSA"] - out["RSL"]
    out["Cumulative Gap"] = out["Gap"].cumsum()
    out["Gap / Total Assets %"] = out["Gap"] / cfg.total_assets * 100
    out["Cum Gap / Total Assets %"] = out["Cumulative Gap"] / cfg.total_assets * 100
    out["Cum Gap / Tier 1 %"] = out["Cumulative Gap"] / cfg.tier1_capital * 100
    out.index.name = "Bucket"
    return out


def gap_by_product(flows: pd.DataFrame) -> pd.DataFrame:
    sign = np.where(flows.side == "A", 1.0, -1.0)
    signed = flows[BL].mul(sign, axis=0)
    signed["product"] = flows["product"].values
    signed["side"] = flows["side"].values
    return signed.groupby(["side", "product"], sort=False)[BL].sum()


gap = gap_table(flows, cfg)
gap_prod = gap_by_product(flows)
gap

,RSA,RSL,Gap,Cumulative Gap,Gap / Total Assets %,Cum Gap / Total Assets %,Cum Gap / Tier 1 %
Bucket,,,,,,,
O/N,"1,525.55","4,310.79","-2,785.24","-2,785.24",-11.14,-11.14,-192.09
O/N-1M,"3,211.33","1,647.25","1,564.08","-1,221.16",6.26,-4.88,-84.22
1M-3M,"4,248.34",987.70,"3,260.65","2,039.49",13.04,8.16,140.65
3M-6M,"1,155.21",652.40,502.81,"2,542.30",2.01,10.17,175.33
6M-9M,"1,145.64",574.77,570.86,"3,113.16",2.28,12.45,214.70
9M-1Y,"1,159.19",660.71,498.48,"3,611.65",1.99,14.45,249.08
1Y-1.5Y,763.13,"1,450.65",-687.52,"2,924.13",-2.75,11.70,201.66
1.5Y-2Y,748.14,"1,218.99",-470.85,"2,453.27",-1.88,9.81,169.19
2Y-3Y,"1,980.90","2,838.25",-857.35,"1,595.92",-3.43,6.38,110.06


In [8]:
by_category = {c: gap_table(flows, cfg, flows.category == c)["Gap"] for c in flows.category.unique()}
pd.DataFrame(by_category).assign(Total=gap["Gap"])

,Liquidity,Loans,Securities,Deposits,Wholesale,Derivatives,Total
Bucket,,,,,,,
O/N,"1,500.00",25.55,0.00,"-4,200.00",-110.79,0.00,"-2,785.24"
O/N-1M,76.43,"2,471.76",0.00,-258.17,"-1,089.21",363.27,"1,564.08"
1M-3M,423.57,"2,987.91",0.00,-487.56,0.00,336.73,"3,260.65"
3M-6M,0.00,"1,155.21",0.00,-652.40,0.00,0.00,502.81
6M-9M,0.00,"1,145.64",0.00,-574.77,0.00,0.00,570.86
9M-1Y,0.00,"1,159.19",0.00,-660.71,0.00,0.00,498.48
1Y-1.5Y,0.00,678.74,84.40,"-1,217.75",0.00,-232.90,-687.52
1.5Y-2Y,0.00,629.79,118.35,"-1,070.15",-148.84,0.00,-470.85
2Y-3Y,0.00,"1,289.94",106.36,"-2,174.64",-450.27,371.25,-857.35


## 7. Scenarios, ΔNII and ΔEVE
**Shocks** follow the BCBS 368 formulas (USD: parallel 200bp, short 300bp, long 150bp) with the rate floor.
**ΔNII (12M, static balance sheet)** = Σ over buckets of gap × shock × (1 − bucket midpoint), for midpoints < 1y; NMD liabilities are scaled by deposit betas.
**ΔEVE** = PV of net repricing flows under shocked vs base zero curve. This is a repricing-flow approximation (principal at repricing date; coupons and commercial margins excluded), so treat it as a screening measure.

In [9]:
BCBS_SCENARIOS = ["Parallel up", "Parallel down", "Steepener", "Flattener", "Short rates up", "Short rates down"]
EXTRA_SCENARIOS = ["Parallel +100bp", "Parallel -100bp"]
ALL_SCENARIOS = BCBS_SCENARIOS + EXTRA_SCENARIOS


def raw_shock(scn: str, t, cfg: Config = cfg) -> np.ndarray:
    t = np.asarray(t, dtype=float)
    P, S, L = cfg.shock_parallel, cfg.shock_short, cfg.shock_long
    short, longf = np.exp(-t / 4), 1 - np.exp(-t / 4)
    table = {
        "Parallel up": P * np.ones_like(t), "Parallel down": -P * np.ones_like(t),
        "Steepener": -0.65 * S * short + 0.90 * L * longf,
        "Flattener": 0.80 * S * short - 0.60 * L * longf,
        "Short rates up": S * short, "Short rates down": -S * short,
        "Parallel +100bp": 0.01 * np.ones_like(t), "Parallel -100bp": -0.01 * np.ones_like(t),
    }
    return table[scn]


def effective_shock(raw: np.ndarray, t, cfg: Config = cfg) -> np.ndarray:
    """Apply BCBS floor to shocked rates; returns effective shock per tenor."""
    base = zero_rate(t, cfg)
    if not cfg.apply_floor:
        return raw
    floor = -0.015 + 0.0003 * np.asarray(t)
    return np.maximum(base + raw, floor) - base


def delta_nii(flows, raw, cfg=cfg, use_beta=None):
    use_beta = cfg.use_deposit_beta if use_beta is None else use_beta
    eff = effective_shock(raw, MID, cfg)
    w = np.clip(1 - MID, 0, None)
    A = flows.loc[flows.side == "A", BL].sum().values
    Lq = flows[flows.side == "L"]
    L_other = Lq.loc[Lq.rate_type != "nmd", BL].sum().values
    total = ((A - L_other) * eff * w).sum()
    for seg, p in cfg.nmd.items():
        Ls = Lq.loc[(Lq.rate_type == "nmd") & (Lq.nmd_segment == seg), BL].sum().values
        beta = np.where(eff >= 0, p["beta_up"], p["beta_down"]) if use_beta else 1.0
        total -= (Ls * beta * eff * w).sum()
    return total


def eve(flows, raw=None, cfg=cfg):
    A = flows.loc[flows.side == "A", BL].sum().values
    L = flows.loc[flows.side == "L", BL].sum().values
    shock = np.zeros(NB) if raw is None else effective_shock(raw, MID, cfg)
    df_ = np.exp(-(zero_rate(MID, cfg) + shock) * MID)
    return ((A - L) * df_).sum()


def baseline_nii(df):
    """12M net interest income on a static balance sheet (incl. derivative legs)."""
    sign = np.where(df.side == "A", 1.0, -1.0)
    return float((sign * df.balance * df.coupon).sum())


def rag(value, limit, cfg=cfg):
    """value/limit are negative-is-bad; returns RED / AMBER / GREEN."""
    if value >= 0:
        return "GREEN"
    use = value / limit
    return "RED" if use > 1 else ("AMBER" if use > cfg.amber_fraction else "GREEN")


def run_scenarios(flows, portfolio, cfg=cfg):
    nii0, eve0 = baseline_nii(portfolio), eve(flows, None, cfg)
    rows = []
    for scn in ALL_SCENARIOS:
        raw = raw_shock(scn, MID, cfg)
        dn, de = delta_nii(flows, raw, cfg), eve(flows, raw, cfg) - eve0
        rows.append(dict(Scenario=scn, BCBS=scn in BCBS_SCENARIOS, dNII=dn, dNII_pct_NII=dn / nii0 * 100,
                         dEVE=de, dEVE_pct_T1=de / cfg.tier1_capital * 100,
                         NII_status=rag(dn / nii0, cfg.nii_limit_pct, cfg), EVE_status=rag(de / cfg.tier1_capital, cfg.eve_limit_pct, cfg)))
    return pd.DataFrame(rows).set_index("Scenario"), nii0, eve0


scen, NII0, EVE0 = run_scenarios(flows, portfolio, cfg)
print(f"Baseline 12M NII: {NII0:,.1f} {cfg.units} | Baseline EVE (rate-sensitive net): {EVE0:,.1f} | Tier 1: {cfg.tier1_capital:,.0f}")
scen

Baseline 12M NII: 751.6 USD mm | Baseline EVE (rate-sensitive net): 456.1 | Tier 1: 1,450


,BCBS,dNII,dNII_pct_NII,dEVE,dEVE_pct_T1,NII_status,EVE_status
Scenario,,,,,,,
Parallel up,True,101.37,13.49,-43.62,-3.01,GREEN,GREEN
Parallel down,True,-107.25,-14.27,114.25,7.88,RED,GREEN
Steepener,True,-97.17,-12.93,-154.47,-10.65,RED,GREEN
Flattener,True,114.33,15.21,160.87,11.09,GREEN,GREEN
Short rates up,True,145.40,19.35,106.33,7.33,GREEN,GREEN
Short rates down,True,-154.14,-20.51,-112.32,-7.75,RED,GREEN
Parallel +100bp,False,50.69,6.74,-29.11,-2.01,GREEN,GREEN
Parallel -100bp,False,-53.62,-7.13,46.63,3.22,GREEN,GREEN


In [10]:
worst_eve = scen.loc[BCBS_SCENARIOS, "dEVE_pct_T1"].min()
worst_nii = scen.loc[BCBS_SCENARIOS, "dNII_pct_NII"].min()
outlier = worst_eve < cfg.eve_limit_pct * 100
print(f"Worst BCBS dEVE / Tier 1: {worst_eve:.2f}%  -> supervisory outlier test (threshold {cfg.eve_limit_pct*100:.0f}%): {'OUTLIER' if outlier else 'PASS'}")
print(f"Worst BCBS dNII / NII:   {worst_nii:.2f}%  -> internal limit {cfg.nii_limit_pct*100:.0f}%")

# Parallel-shock ladder and impact of deposit betas
ladder = []
for bp in range(-300, 301, 50):
    raw = np.full(NB, bp / 10000)
    ladder.append(dict(shock_bp=bp, dNII_with_beta=delta_nii(flows, raw, cfg, True), dNII_no_beta=delta_nii(flows, raw, cfg, False),
                       dEVE=eve(flows, raw, cfg) - EVE0))
ladder = pd.DataFrame(ladder).set_index("shock_bp")
ladder

Worst BCBS dEVE / Tier 1: -10.65%  -> supervisory outlier test (threshold -15%): PASS
Worst BCBS dNII / NII:   -20.51%  -> internal limit -10%


,dNII_with_beta,dNII_no_beta,dEVE
shock_bp,,,
-300,-160.87,-60.87,206.99
-250,-134.06,-50.73,157.20
-200,-107.25,-40.58,114.25
-150,-80.44,-30.44,77.58
-100,-53.62,-20.29,46.63
-50,-26.81,-10.15,20.92
0,0.00,0.00,0.00
50,25.34,10.15,-16.55
100,50.69,20.29,-29.11


## 8. Visualisation

In [11]:
def plot_gap(gap):
    fig = make_subplots(specs=[[{"secondary_y": False}]])
    fig.add_bar(x=gap.index, y=gap["RSA"], name="RSA", marker_color="#1f77b4")
    fig.add_bar(x=gap.index, y=-gap["RSL"], name="-RSL", marker_color="#d62728")
    fig.add_scatter(x=gap.index, y=gap["Gap"], name="Gap", mode="lines+markers", line=dict(color="black", dash="dot"))
    fig.add_scatter(x=gap.index, y=gap["Cumulative Gap"], name="Cumulative gap", mode="lines+markers", line=dict(color="#2ca02c", width=3))
    fig.update_layout(title=f"Repricing gap by BCBS bucket ({cfg.units})", barmode="relative", template="plotly_white", height=520, xaxis_title="Time bucket")
    return fig


def plot_heatmap(gp):
    fig = go.Figure(go.Heatmap(z=gp.values, x=BL, y=[p for _, p in gp.index], colorscale="RdBu", zmid=0, colorbar=dict(title=cfg.units)))
    fig.update_layout(title="Net repricing by product and bucket (assets +, liabilities -)", height=620, template="plotly_white")
    return fig


def plot_scenarios(scen):
    fig = make_subplots(rows=1, cols=2, subplot_titles=("ΔNII, % of baseline NII", "ΔEVE, % of Tier 1"))
    colors = lambda s: ["#d62728" if v < 0 else "#2ca02c" for v in s]
    fig.add_bar(y=scen.index, x=scen["dNII_pct_NII"], orientation="h", marker_color=colors(scen["dNII_pct_NII"]), row=1, col=1, showlegend=False)
    fig.add_bar(y=scen.index, x=scen["dEVE_pct_T1"], orientation="h", marker_color=colors(scen["dEVE_pct_T1"]), row=1, col=2, showlegend=False)
    fig.add_vline(x=cfg.nii_limit_pct * 100, line_dash="dash", line_color="red", row=1, col=1)
    fig.add_vline(x=cfg.eve_limit_pct * 100, line_dash="dash", line_color="red", row=1, col=2)
    fig.update_layout(height=480, template="plotly_white", title="Scenario results vs limits")
    return fig


def plot_shocks():
    fig = go.Figure()
    for scn in BCBS_SCENARIOS:
        fig.add_scatter(x=BL, y=effective_shock(raw_shock(scn, MID), MID) * 1e4, name=scn, mode="lines+markers")
    fig.update_layout(title="BCBS 368 shock profiles (bp, after floor)", template="plotly_white", height=420, yaxis_title="bp")
    return fig


def plot_ladder(ladder):
    fig = go.Figure()
    fig.add_scatter(x=ladder.index, y=ladder["dNII_with_beta"], name="ΔNII (with betas)", mode="lines+markers")
    fig.add_scatter(x=ladder.index, y=ladder["dNII_no_beta"], name="ΔNII (beta = 100%)", mode="lines+markers", line=dict(dash="dash"))
    fig.add_scatter(x=ladder.index, y=ladder["dEVE"], name="ΔEVE", mode="lines+markers")
    fig.update_layout(title="Parallel shock ladder", xaxis_title="Shock (bp)", yaxis_title=cfg.units, template="plotly_white", height=420)
    return fig


for f in (plot_gap(gap), plot_heatmap(gap_prod), plot_scenarios(scen), plot_shocks(), plot_ladder(ladder)):
    f.show()

## 9. Export

In [12]:
def export_results(cfg=cfg):
    out = Path(cfg.output_dir)
    xlsx = out / f"irrbb_report_{cfg.report_date}.xlsx"
    summary = pd.DataFrame({
        "Metric": ["Report date", "Baseline 12M NII", "Tier 1 capital", "Worst BCBS dEVE % Tier 1", "EVE outlier test", "Worst BCBS dNII % NII"],
        "Value": [cfg.report_date, round(NII0, 2), cfg.tier1_capital, round(worst_eve, 2), "OUTLIER" if outlier else "PASS", round(worst_nii, 2)]})
    assumptions = pd.json_normalize(asdict(cfg), sep=".").T.reset_index()
    assumptions.columns = ["Parameter", "Value"]
    assumptions["Value"] = assumptions["Value"].astype(str)
    with pd.ExcelWriter(xlsx, engine="openpyxl") as xw:
        summary.to_excel(xw, "Summary", index=False)
        gap.to_excel(xw, "Gap")
        gap_prod.to_excel(xw, "Gap by product")
        scen.to_excel(xw, "Scenarios")
        ladder.to_excel(xw, "Shock ladder")
        exceptions.to_excel(xw, "Exceptions", index=False)
        assumptions.to_excel(xw, "Assumptions", index=False)
        bs.to_excel(xw, "Balance sheet", index=False)
    portfolio.to_csv(out / "portfolio.csv", index=False)
    log.info("Exported to %s", out.resolve())
    return xlsx

xlsx_path = export_results()
try:
    from google.colab import files
    files.download(str(xlsx_path))
except Exception:
    pass

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 10. Automated model tests

In [13]:
def run_tests():
    results = {}
    nd = portfolio[~portfolio.is_derivative]

    results["Balance sheet balances"] = abs(nd.loc[nd.side == "A", "balance"].sum() - nd.loc[nd.side == "L", "balance"].sum()) < 1e-6
    sens = portfolio[portfolio.rate_sensitive].reset_index(drop=True)
    results["Flows conserve balance (per instrument)"] = np.allclose(flows[BL].sum(axis=1).values, sens["balance"].values, atol=1e-8)
    results["No negative flows"] = (flows[BL].values >= -1e-12).all()
    sa = sens.loc[sens.side == "A", "balance"].sum(); sl = sens.loc[sens.side == "L", "balance"].sum()
    results["Sum of gaps = sensitive A - L"] = abs(gap["Gap"].sum() - (sa - sl)) < 1e-6
    results["Cumulative gap ends at sum of gaps"] = abs(gap["Cumulative Gap"].iloc[-1] - gap["Gap"].sum()) < 1e-9
    results["Zero shock -> zero dNII"] = abs(delta_nii(flows, np.zeros(NB))) < 1e-12
    results["Zero shock -> zero dEVE"] = abs(eve(flows, np.zeros(NB)) - EVE0) < 1e-9
    results["Bucket edges correct"] = list(assign_bucket([0.0, 1/365.25, 1/12, 0.25, 1.0, 10.0, 20.0, 40.0])) == [0, 0, 1, 2, 5, 15, 17, 18]
    results["dNII linear in small up shock (no floor)"] = abs(delta_nii(flows, np.full(NB, 0.02)) - 2 * delta_nii(flows, np.full(NB, 0.01))) < 1e-6
    results["NMD within BCBS caps"] = all(p["core"] <= p["cap_core"] and p["avg_mat"] <= p["cap_mat"] for p in cfg.nmd.values())
    results["Beta lowers NII sensitivity to +200bp"] = abs(delta_nii(flows, np.full(NB, 0.02), cfg, True)) != abs(delta_nii(flows, np.full(NB, 0.02), cfg, False))
    results["Floor binds in extreme down shock"] = effective_shock(np.full(NB, -0.10), MID)[0] > -0.10
    res = pd.Series(results, name="passed")
    assert res.all(), f"Failed tests:\n{res[~res]}"
    return res

pd.DataFrame(run_tests().map({True: "PASS", False: "FAIL"}))

,passed
Balance sheet balances,PASS
Flows conserve balance (per instrument),PASS
No negative flows,PASS
Sum of gaps = sensitive A - L,PASS
Cumulative gap ends at sum of gaps,PASS
Zero shock -> zero dNII,PASS
Zero shock -> zero dEVE,PASS
Bucket edges correct,PASS
dNII linear in small up shock (no floor),PASS
NMD within BCBS caps,PASS


## 11. Methodology notes and limitations
- **Static balance sheet**: maturing and repricing balances are not rolled over with new business, so ΔNII isolates repricing risk only.
- **Gap-based ΔNII**: uses bucket midpoints as repricing times and ignores basis risk between indices, caps/floors on loans, and optionality beyond the CPR and early-withdrawal overlays.
- **NMD**: a single core share and uniform ladder per segment; calibrate to observed balance and rate history for production use. Betas apply to the whole NMD balance.
- **ΔEVE**: principal repricing flows discounted at the risk-free zero curve; excludes coupon cash flows and commercial margins, and does not include automatic-option (e.g. embedded floors) valuation. For a full BCBS standardised calculation add coupon flows and spread-discounting.
- **Single currency**: multi-currency requires per-currency shock parameters and the BCBS aggregation rule.
- **Upload mode**: set `use_uploaded_file = True`. Required columns: `instrument_id, product, side, rate_type, balance, coupon`; optional: `category, amort, rem_months` (or `maturity_date`), `reset_days` (or `next_reset_date`), `cpr, nmd_segment, is_derivative`.